In [1]:
# ==================================================
# RECUPERAR CONEXIÓN CON SQL SERVER
# ==================================================

import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text
from dotenv import load_dotenv

# Carpeta principal del proyecto
RUTA_PROYECTO = Path.cwd()

# Cargar configuración del proyecto
load_dotenv(RUTA_PROYECTO / ".env")

# Localizar el archivo database.py
RUTA_SRC = RUTA_PROYECTO / "src"

if str(RUTA_SRC) not in sys.path:
    sys.path.insert(0, str(RUTA_SRC))

# Importar conexión original
from database import engine_sqlserver

# Conectar con SQL Server
engine = engine_sqlserver()

with engine.connect() as conexion:
    nombre_bd = conexion.execute(
        text("SELECT DB_NAME()")
    ).scalar_one()

print("Conexión establecida correctamente.")
print("Base de datos:", nombre_bd)


Conexión establecida correctamente.
Base de datos: RedSaludCarazoDW_V2


In [2]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# EVALUACIÓN DE LOS 3 KPI DE CALIDAD DE DATOS
# ============================================================

import pandas as pd
from sqlalchemy import text
from IPython.display import display

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")


# ============================================================
# 1. CONSULTAR LOS DATOS DESDE SQL SERVER
# ============================================================

with engine.connect() as conexion:

    # Registros originales
    datos = pd.read_sql_query(
        text("SELECT * FROM dbo.raw_cita"),
        conexion
    )

    # Registros preparados para minería de datos
    datos_predictivos = pd.read_sql_query(
        text("SELECT * FROM dbo.dm_dataset_prediccion"),
        conexion
    )

total_registros = len(datos)
total_predictivos = len(datos_predictivos)

if total_registros == 0:
    raise ValueError(
        "No existen registros originales para evaluar."
    )

print("\nDATOS CONSULTADOS")
print("Registros originales:", total_registros)
print("Registros predictivos:", total_predictivos)


# ============================================================
# KPI 1. PORCENTAJE DE REGISTROS VALIDADOS (PRV)
# ============================================================
#
# Fórmula:
# PRV = (Registros en dataset predictivo /
#        Total de registros originales) * 100
#
# Meta: >= 90%
#
# Este KPI mide la proporción incorporada al dataset
# predictivo. No demuestra por sí solo que todas
# las exclusiones sean errores de calidad.
# ============================================================

registros_validados = total_predictivos

prv = (
    registros_validados / total_registros
) * 100

meta_prv = 90

estado_prv = (
    "CUMPLE"
    if prv >= meta_prv
    else "NO CUMPLE"
)


# ============================================================
# KPI 2. TASA DE INTEGRIDAD DE REGISTROS DE CITAS
# ============================================================
#
# Fórmula:
# TID = (Registros completos /
#        Total de registros procesados) * 100
#
# Meta del II corte: > 98%
#
# ============================================================

campos_obligatorios = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad",
    "espera_dias",
    "citas_previas_perdidas",
    "recordatorios_enviados",
    "no_asistio"
]

# Verificar existencia de columnas
faltan_columnas = (
    set(campos_obligatorios) - set(datos.columns)
)

if faltan_columnas:
    raise ValueError(
        f"Faltan columnas: {sorted(faltan_columnas)}"
    )

# Detectar valores nulos
faltantes = datos[campos_obligatorios].isna().copy()

# Detectar también textos vacíos
for campo in campos_obligatorios:

    serie = datos[campo]

    if (
        pd.api.types.is_object_dtype(serie)
        or pd.api.types.is_string_dtype(serie)
    ):

        vacios = (
            serie.astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
        )

        faltantes[campo] = (
            faltantes[campo] | vacios
        )

# Un registro es incompleto si falta
# al menos uno de los campos obligatorios
registros_incompletos = int(
    faltantes.any(axis=1).sum()
)

registros_completos = (
    total_registros - registros_incompletos
)

tasa_integridad = (
    registros_completos / total_registros
) * 100

meta_integridad = 98

estado_integridad = (
    "CUMPLE"
    if tasa_integridad > meta_integridad
    else "NO CUMPLE"
)


# ============================================================
# KPI 3. TASA DE DUPLICIDAD DE IDENTIFICADORES DE CITA
# ============================================================
#
# Fórmula del II corte:
# (IDs de citas repetidos / Total de registros) * 100
#
# Se cuentan los identificadores distintos
# que aparecen más de una vez.
#
# Meta: 0%
#
# ============================================================

ids = (
    datos["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

# Contar frecuencia de cada identificador
frecuencias = ids.dropna().value_counts()

# Identificadores que aparecen más de una vez
ids_repetidos = int(
    (frecuencias > 1).sum()
)

# Filas involucradas en los grupos duplicados
filas_duplicadas = int(
    frecuencias[frecuencias > 1].sum()
)

tasa_duplicidad = (
    ids_repetidos / total_registros
) * 100

meta_duplicidad = 0

estado_duplicidad = (
    "CUMPLE"
    if ids_repetidos == 0
    else "NO CUMPLE"
)


# ============================================================
# 4. CONSTRUIR TABLA GENERAL DE KPI
# ============================================================

tabla_3_kpis = pd.DataFrame([
    {
        "Código": "PRV",
        "KPI": "Porcentaje de Registros Validados",
        "Resultado (%)": round(prv, 2),
        "Meta": ">=90%",
        "Estado": estado_prv
    },
    {
        "Código": "IND-01",
        "KPI": "Tasa de integridad de registros de citas",
        "Resultado (%)": round(tasa_integridad, 2),
        "Meta": ">98%",
        "Estado": estado_integridad
    },
    {
        "Código": "IND-05",
        "KPI": "Tasa de duplicidad de identificadores de cita",
        "Resultado (%)": round(tasa_duplicidad, 2),
        "Meta": "0%",
        "Estado": estado_duplicidad
    }
])


# ============================================================
# 5. MOSTRAR RESULTADOS
# ============================================================

print("\n" + "=" * 70)
print("DETALLE DE LOS INDICADORES")
print("=" * 70)

print("\nKPI 1. PORCENTAJE DE REGISTROS VALIDADOS")
print("Registros originales:", total_registros)
print("Registros predictivos:", registros_validados)
print(f"Resultado: {prv:.2f}%")
print("Estado:", estado_prv)

print("\nKPI 2. TASA DE INTEGRIDAD")
print("Registros completos:", registros_completos)
print("Registros incompletos:", registros_incompletos)
print(f"Resultado: {tasa_integridad:.2f}%")
print("Estado:", estado_integridad)

print("\nKPI 3. TASA DE DUPLICIDAD")
print("Identificadores repetidos:", ids_repetidos)
print("Filas involucradas:", filas_duplicadas)
print(f"Resultado: {tasa_duplicidad:.2f}%")
print("Estado:", estado_duplicidad)

print("\n" + "=" * 70)
print("TABLA GENERAL DE LOS 3 KPI DE CALIDAD")
print("=" * 70)

display(tabla_3_kpis)

print("\nEvaluación finalizada.")
print("No se modificaron los datos de SQL Server.")


RED SALUD INTEGRAL CARAZO

DATOS CONSULTADOS
Registros originales: 10080
Registros predictivos: 8996

DETALLE DE LOS INDICADORES

KPI 1. PORCENTAJE DE REGISTROS VALIDADOS
Registros originales: 10080
Registros predictivos: 8996
Resultado: 89.25%
Estado: NO CUMPLE

KPI 2. TASA DE INTEGRIDAD
Registros completos: 9160
Registros incompletos: 920
Resultado: 90.87%
Estado: NO CUMPLE

KPI 3. TASA DE DUPLICIDAD
Identificadores repetidos: 80
Filas involucradas: 160
Resultado: 0.79%
Estado: NO CUMPLE

TABLA GENERAL DE LOS 3 KPI DE CALIDAD


,Código,KPI,Resultado (%),Meta,Estado
0,PRV,Porcentaje de Registros Validados,89.25,>=90%,NO CUMPLE
1,IND-01,Tasa de integridad de registros de citas,90.87,>98%,NO CUMPLE
2,IND-05,Tasa de duplicidad de identificadores de cita,0.79,0%,NO CUMPLE



Evaluación finalizada.
No se modificaron los datos de SQL Server.


In [3]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# PASO 36. PRUEBAS DE LAS CUATRO POLÍTICAS
# Casos válidos e inválidos
# ============================================================

import pandas as pd
import json
import sys
import importlib

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. LOCALIZAR EL PROYECTO Y CARGAR LAS POLÍTICAS
# ============================================================

rutas_posibles = [
    Path.cwd(),
    Path.cwd() / "RedSaludCarazo_proyecto_base",
    Path.cwd().parent
]

proyecto = next(
    (
        ruta for ruta in rutas_posibles
        if (ruta / "src" / "politicas_gobernanza.py").is_file()
    ),
    None
)

if proyecto is None:
    raise FileNotFoundError(
        "No se encontró src/politicas_gobernanza.py"
    )

sys.path.insert(0, str(proyecto / "src"))

import politicas_gobernanza

importlib.reload(politicas_gobernanza)


# ============================================================
# 2. CARGAR CATÁLOGOS PROVISIONALES
# ============================================================

ruta_catalogos = (
    proyecto / "catalogos_propuestos_gobernanza.json"
)

with open(
    ruta_catalogos,
    "r",
    encoding="utf-8"
) as archivo:

    catalogos = json.load(archivo)


# ============================================================
# 3. CREAR UN REGISTRO DE PRUEBA VÁLIDO
# ============================================================

registro_base = {
    "id_cita": "PRUEBA-001",
    "fecha_cita": "2026-06-15",
    "centro_salud": catalogos["centro_salud"][0],
    "especialidad": catalogos["especialidad"][0],
    "canal_agendamiento": catalogos["canal_agendamiento"][0],
    "grupo_edad": "45-59",
    "espera_dias": 5,
    "citas_previas_perdidas": 0,
    "recordatorios_enviados": 2,
    "no_asistio": 0
}

registro_valido = pd.DataFrame([registro_base])


# ============================================================
# 4. PREPARAR LOS CASOS DE PRUEBA
# ============================================================

escenarios = []

# ------------------------------------------------------------
# POL-02. COMPLETITUD
# ------------------------------------------------------------

# Caso válido: campos completos
escenarios.append((
    "POL-02",
    "VÁLIDO",
    registro_valido.copy(),
    0,
    "Todos los campos completos"
))

# Caso inválido: centro de salud faltante
pol02_invalido = registro_valido.copy()

pol02_invalido.loc[0, "centro_salud"] = None

escenarios.append((
    "POL-02",
    "INVÁLIDO",
    pol02_invalido,
    1,
    "Centro de salud vacío"
))


# ------------------------------------------------------------
# POL-03. ESTANDARIZACIÓN
# ------------------------------------------------------------

# Caso válido: categorías del catálogo
escenarios.append((
    "POL-03",
    "VÁLIDO",
    registro_valido.copy(),
    0,
    "Categorías del catálogo provisional"
))

# Caso inválido: centro inexistente en el catálogo
pol03_invalido = registro_valido.copy()

pol03_invalido.loc[0, "centro_salud"] = (
    "CENTRO NO REGISTRADO"
)

escenarios.append((
    "POL-03",
    "INVÁLIDO",
    pol03_invalido,
    1,
    "Centro fuera del catálogo"
))


# ------------------------------------------------------------
# POL-04. DUPLICIDAD
# ------------------------------------------------------------

# Caso válido: dos identificadores diferentes
pol04_valido = pd.concat(
    [registro_valido, registro_valido],
    ignore_index=True
)

pol04_valido.loc[1, "id_cita"] = "PRUEBA-002"

escenarios.append((
    "POL-04",
    "VÁLIDO",
    pol04_valido,
    0,
    "Dos identificadores diferentes"
))

# Caso inválido: dos citas con el mismo identificador
pol04_invalido = pol04_valido.copy()

pol04_invalido.loc[1, "id_cita"] = "PRUEBA-001"

escenarios.append((
    "POL-04",
    "INVÁLIDO",
    pol04_invalido,
    1,
    "Un grupo de identificadores repetidos"
))


# ------------------------------------------------------------
# POL-05. VALORES ANÓMALOS
# ------------------------------------------------------------

# Caso válido: días de espera positivos
escenarios.append((
    "POL-05",
    "VÁLIDO",
    registro_valido.copy(),
    0,
    "Días de espera = 5"
))

# Caso inválido: días de espera negativos
pol05_invalido = registro_valido.copy()

pol05_invalido.loc[0, "espera_dias"] = -5

escenarios.append((
    "POL-05",
    "INVÁLIDO",
    pol05_invalido,
    1,
    "Días de espera = -5"
))


# ============================================================
# 5. EJECUTAR LAS OCHO PRUEBAS
# ============================================================

resultados_pruebas = []

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("PRUEBAS DE POLÍTICAS DE GOBERNANZA")
print("=" * 70)

for politica, tipo, datos_prueba, esperado, descripcion in escenarios:

    # Ejecutar el validador original
    resultado = politicas_gobernanza.evaluar_politicas(
        datos_prueba,
        catalogos=catalogos
    )

    # Recuperar únicamente la política evaluada
    fila = resultado.loc[
        resultado["Politica"] == politica
    ].iloc[0]

    obtenido = int(fila["Incumplimientos"])

    # Comprobar si respondió como esperábamos
    prueba_superada = obtenido == esperado

    resultados_pruebas.append({
        "Política": politica,
        "Caso": tipo,
        "Dato probado": descripcion,
        "Esperado": esperado,
        "Detectado": obtenido,
        "Estado del validador": fila["Estado"],
        "Resultado prueba": (
            "SUPERADA"
            if prueba_superada
            else "FALLIDA"
        )
    })


# ============================================================
# 6. PRESENTAR TABLA DE PRUEBAS
# ============================================================

tabla_pruebas = pd.DataFrame(
    resultados_pruebas
)

print("\nRESULTADOS DE LAS PRUEBAS")
display(tabla_pruebas)

total_pruebas = len(tabla_pruebas)

pruebas_superadas = int(
    (tabla_pruebas["Resultado prueba"] == "SUPERADA").sum()
)

print("\nRESUMEN")
print("Pruebas ejecutadas:", total_pruebas)
print("Pruebas superadas:", pruebas_superadas)
print("Pruebas fallidas:", total_pruebas - pruebas_superadas)


# ============================================================
# 7. GUARDAR EVIDENCIA DE AUDITORÍA
# ============================================================

carpeta_resultados = proyecto / "resultados_jupyter"

carpeta_resultados.mkdir(
    parents=True,
    exist_ok=True
)

fecha_ejecucion = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

archivo_resultados = (
    carpeta_resultados
    / f"pruebas_politicas_{fecha_ejecucion}.csv"
)

tabla_pruebas.to_csv(
    archivo_resultados,
    index=False,
    encoding="utf-8-sig"
)

print("\nEVIDENCIA GUARDADA:")
print(archivo_resultados)

if pruebas_superadas == total_pruebas:
    print("\nRESULTADO GENERAL: TODAS LAS PRUEBAS SUPERADAS")
else:
    print("\nRESULTADO GENERAL: EXISTEN PRUEBAS FALLIDAS")

print("\nNo se modificaron registros de SQL Server.")


RED SALUD INTEGRAL CARAZO
PRUEBAS DE POLÍTICAS DE GOBERNANZA

RESULTADOS DE LAS PRUEBAS


,Política,Caso,Dato probado,Esperado,Detectado,Estado del validador,Resultado prueba
0,POL-02,VÁLIDO,Todos los campos completos,0,0,SIN HALLAZGOS,SUPERADA
1,POL-02,INVÁLIDO,Centro de salud vacío,1,1,CON HALLAZGOS,SUPERADA
2,POL-03,VÁLIDO,Categorías del catálogo provisional,0,0,SIN HALLAZGOS - CATÁLOGO PROVISIONAL,SUPERADA
3,POL-03,INVÁLIDO,Centro fuera del catálogo,1,1,CON HALLAZGOS,SUPERADA
4,POL-04,VÁLIDO,Dos identificadores diferentes,0,0,SIN HALLAZGOS,SUPERADA
5,POL-04,INVÁLIDO,Un grupo de identificadores repetidos,1,1,REQUIERE REVISIÓN,SUPERADA
6,POL-05,VÁLIDO,Días de espera = 5,0,0,SIN HALLAZGOS,SUPERADA
7,POL-05,INVÁLIDO,Días de espera = -5,1,1,CON HALLAZGOS,SUPERADA



RESUMEN
Pruebas ejecutadas: 8
Pruebas superadas: 8
Pruebas fallidas: 0

EVIDENCIA GUARDADA:
c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\pruebas_politicas_20261021T160333Z.csv

RESULTADO GENERAL: TODAS LAS PRUEBAS SUPERADAS

No se modificaron registros de SQL Server.


In [4]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# PASO 37. REVISIÓN DEL REGISTRO DE ACCIONES PHVA
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

# Buscar el archivo dentro del proyecto
rutas_posibles = [
    Path.cwd() / "resultados_jupyter" / "registro_acciones_phva.csv",
    Path.cwd() / "RedSaludCarazo_proyecto_base" / "resultados_jupyter" / "registro_acciones_phva.csv",
    Path.cwd().parent / "resultados_jupyter" / "registro_acciones_phva.csv"
]

ruta_phva = next(
    (ruta for ruta in rutas_posibles if ruta.is_file()),
    None
)

if ruta_phva is None:
    raise FileNotFoundError(
        "No se encontró registro_acciones_phva.csv "
        "en la carpeta resultados_jupyter."
    )

# Leer el registro existente
acciones_phva = pd.read_csv(ruta_phva)

print("=" * 65)
print("RED SALUD INTEGRAL CARAZO")
print("REGISTRO DE ACCIONES CORRECTIVAS - CICLO PHVA")
print("=" * 65)

print("\nArchivo:", ruta_phva.name)
print("Acciones registradas:", len(acciones_phva))

print("\nCOLUMNAS DISPONIBLES:")
print(", ".join(acciones_phva.columns))

print("\nREGISTRO DE ACCIONES:")
display(acciones_phva)

print("\nLa consulta no modificó el archivo original.")


RED SALUD INTEGRAL CARAZO
REGISTRO DE ACCIONES CORRECTIVAS - CICLO PHVA

Archivo: registro_acciones_phva.csv
Acciones registradas: 4

COLUMNAS DISPONIBLES:
Codigo, Hallazgo, Cantidad, Politica, Accion_propuesta, Responsable_propuesto, Estado, Verificacion, Fecha_revision_UTC

REGISTRO DE ACCIONES:


,Codigo,Hallazgo,Cantidad,Politica,Accion_propuesta,Responsable_propuesto,Estado,Verificacion,Fecha_revision_UTC
0,H01,Registros incompletos,920,POL-02 Completitud,Revisar los campos faltantes y su captura,Admisión y TI,PENDIENTE,Recalcular el KPI de completitud,2026-10-16T21:21:27.618461+00:00
1,H02,Valores lógicos inválidos,3,POL-05 Valores anómalos,Revisar los registros y las reglas incumplidas,TI,PENDIENTE,Repetir las validaciones de rangos y valores,2026-10-16T21:21:27.618461+00:00
2,H03,Identificadores repetidos,80,POL-04 Duplicidad,Comprobar los grupos repetidos antes de corregir,TI,PENDIENTE,Repetir el control de unicidad,2026-10-16T21:21:27.618461+00:00
3,H04,Sin correspondencia predictiva,1084,Revisión de reglas y linaje,Investigar por qué no aparecen en el dataset,TI y calidad,PENDIENTE,Comparar nuevamente origen y destino,2026-10-16T21:21:27.618461+00:00



La consulta no modificó el archivo original.


In [6]:

# ============================================================
# PASO 38. DEMOSTRACIÓN DEL CICLO PHVA
# RED SALUD INTEGRAL CARAZO
# POL-02: COMPLETITUD DE REGISTROS
# ============================================================

import pandas as pd
import json
import sys
import importlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

# ------------------------------------------------------------
# 1. CARGAR LAS POLÍTICAS Y LOS CATÁLOGOS DEL PROYECTO
# ------------------------------------------------------------

rutas = [
    Path.cwd(),
    Path.cwd() / "RedSaludCarazo_proyecto_base",
    Path.cwd().parent
]

proyecto = next(
    (
        ruta for ruta in rutas
        if (ruta / "src" / "politicas_gobernanza.py").is_file()
    ),
    None
)

if proyecto is None:
    raise FileNotFoundError(
        "No se encontró politicas_gobernanza.py"
    )

sys.path.insert(0, str(proyecto / "src"))

import politicas_gobernanza
importlib.reload(politicas_gobernanza)

with open(
    proyecto / "catalogos_propuestos_gobernanza.json",
    encoding="utf-8"
) as archivo:
    catalogos = json.load(archivo)


# ------------------------------------------------------------
# 2. CREAR UN REGISTRO DE PRUEBA
# ------------------------------------------------------------

registro = pd.DataFrame([{
    "id_cita": "PHVA-001",
    "fecha_cita": "2026-06-15",
    "centro_salud": None,
    "especialidad": catalogos["especialidad"][0],
    "canal_agendamiento": catalogos["canal_agendamiento"][0],
    "grupo_edad": "45-59",
    "espera_dias": 5,
    "citas_previas_perdidas": 0,
    "recordatorios_enviados": 2,
    "no_asistio": 0
}])


def verificar_pol02(datos):
    resultado = politicas_gobernanza.evaluar_politicas(
        datos,
        catalogos=catalogos
    )

    return int(
        resultado.loc[
            resultado["Politica"] == "POL-02",
            "Incumplimientos"
        ].iloc[0]
    )


print("=" * 65)
print("DEMOSTRACIÓN PHVA - POL-02")
print("=" * 65)


# ------------------------------------------------------------
# PLANIFICAR: IDENTIFICAR EL PROBLEMA
# ------------------------------------------------------------

incumplimientos_antes = verificar_pol02(registro)

print("\n1. PLANIFICAR")
print("Problema: falta el centro de salud.")
print("Incumplimientos:", incumplimientos_antes)
print("Acción: completar el campo faltante.")


# ------------------------------------------------------------
# HACER: CORREGIR EL REGISTRO DE PRUEBA
# ------------------------------------------------------------

registro_corregido = registro.copy()

# Valor conocido del catálogo de prueba
registro_corregido.loc[0, "centro_salud"] = (
    catalogos["centro_salud"][0]
)

print("\n2. HACER")
print("Se completó el centro de salud")
print("en una copia del registro de prueba.")


# ------------------------------------------------------------
# VERIFICAR: EJECUTAR NUEVAMENTE LA POLÍTICA
# ------------------------------------------------------------

incumplimientos_despues = verificar_pol02(
    registro_corregido
)

print("\n3. VERIFICAR")
print("Incumplimientos antes:", incumplimientos_antes)
print("Incumplimientos después:", incumplimientos_despues)


# ------------------------------------------------------------
# ACTUAR: PROBAR UN CONTROL PREVENTIVO
# ------------------------------------------------------------

def permitir_registro(datos):
    return verificar_pol02(datos) == 0

rechaza_incompleto = not permitir_registro(registro)
acepta_completo = permitir_registro(registro_corregido)

print("\n4. ACTUAR")
print(
    "Registro incompleto rechazado:",
    rechaza_incompleto
)
print(
    "Registro completo aceptado:",
    acepta_completo
)

# ------------------------------------------------------------
# CONSTRUIR LA EVIDENCIA
# ------------------------------------------------------------

prueba_superada = (
    incumplimientos_antes == 1
    and incumplimientos_despues == 0
    and rechaza_incompleto
    and acepta_completo
)

tabla_phva = pd.DataFrame([
    {
        "Etapa": "PLANIFICAR",
        "Acción": "Identificar campo faltante",
        "Resultado": f"{incumplimientos_antes} incumplimiento"
    },
    {
        "Etapa": "HACER",
        "Acción": "Completar registro de prueba",
        "Resultado": "Corrección simulada"
    },
    {
        "Etapa": "VERIFICAR",
        "Acción": "Repetir POL-02",
        "Resultado": f"{incumplimientos_despues} incumplimientos"
    },
    {
        "Etapa": "ACTUAR",
        "Acción": "Comprobar control preventivo",
        "Resultado": (
            "Prueba superada"
            if rechaza_incompleto and acepta_completo
            else "Requiere revisión"
        )
    }
])

print("\n" + "=" * 65)
print("EVIDENCIA DEL CICLO PHVA")
print("=" * 65)

display(tabla_phva)

print(
    "\nRESULTADO:",
    "PRUEBA SUPERADA" if prueba_superada
    else "REQUIERE REVISIÓN"
)

# ------------------------------------------------------------
# GUARDAR EVIDENCIA DE LA PRUEBA
# ------------------------------------------------------------

carpeta = proyecto / "resultados_jupyter"
carpeta.mkdir(parents=True, exist_ok=True)

fecha = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

ruta_salida = (
    carpeta / f"prueba_phva_pol02_{fecha}.csv"
)

tabla_phva.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivo generado:", ruta_salida.name)
print("Tipo de evidencia: simulación controlada.")
print("No se modificaron registros de SQL Server.")
print("Las acciones institucionales siguen pendientes.")


DEMOSTRACIÓN PHVA - POL-02

1. PLANIFICAR
Problema: falta el centro de salud.
Incumplimientos: 1
Acción: completar el campo faltante.

2. HACER
Se completó el centro de salud
en una copia del registro de prueba.

3. VERIFICAR
Incumplimientos antes: 1
Incumplimientos después: 0

4. ACTUAR
Registro incompleto rechazado: True
Registro completo aceptado: True

EVIDENCIA DEL CICLO PHVA


,Etapa,Acción,Resultado
0,PLANIFICAR,Identificar campo faltante,1 incumplimiento
1,HACER,Completar registro de prueba,Corrección simulada
2,VERIFICAR,Repetir POL-02,0 incumplimientos
3,ACTUAR,Comprobar control preventivo,Prueba superada



RESULTADO: PRUEBA SUPERADA

Archivo generado: prueba_phva_pol02_20261021T161335Z.csv
Tipo de evidencia: simulación controlada.
No se modificaron registros de SQL Server.
Las acciones institucionales siguen pendientes.


# RESULTADOS FINALES
## Calidad y Gobernanza de Datos — Red Salud Integral Carazo

Evidencia de KPI, políticas programadas, auditoría,
pruebas de validación y ciclo PHVA.

In [4]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# PASO 39. RESUMEN FINAL DE CALIDAD Y GOBERNANZA
# ============================================================

import sys
import json
import importlib
import pandas as pd

from pathlib import Path
from sqlalchemy import text
from dotenv import load_dotenv
from IPython.display import display


# ============================================================
# 1. LOCALIZAR EL PROYECTO
# ============================================================

rutas = [
    Path.cwd(),
    Path.cwd() / "RedSaludCarazo_proyecto_base",
    Path.cwd().parent
]

proyecto = next(
    (
        ruta for ruta in rutas
        if (ruta / "src" / "database.py").is_file()
        and (ruta / "src" / "politicas_gobernanza.py").is_file()
    ),
    None
)

if proyecto is None:
    raise FileNotFoundError(
        "No se encontró la carpeta del proyecto."
    )

sys.path.insert(0, str(proyecto / "src"))

load_dotenv(
    proyecto / ".env",
    override=True
)

from database import engine_sqlserver
import politicas_gobernanza

importlib.reload(politicas_gobernanza)

engine = engine_sqlserver()

carpeta_resultados = proyecto / "resultados_jupyter"


# ============================================================
# 2. FUNCIÓN PARA LEER LAS EVIDENCIAS
# ============================================================

def leer_ultimo_csv(patron):

    archivos = list(
        carpeta_resultados.glob(patron)
    )

    if not archivos:
        raise FileNotFoundError(
            f"No se encontró un archivo para: {patron}"
        )

    archivo = max(
        archivos,
        key=lambda ruta: ruta.stat().st_mtime
    )

    return pd.read_csv(archivo)


# ============================================================
# 3. CONSULTAR DATOS DESDE SQL SERVER
# ============================================================

with engine.connect() as conexion:

    datos = pd.read_sql_query(
        text("""
            SELECT *
            FROM dbo.raw_cita
        """),
        conexion
    )

    predictivo = pd.read_sql_query(
        text("""
            SELECT id_cita
            FROM dbo.dm_dataset_prediccion
        """),
        conexion
    )

total = len(datos)

if total == 0:
    raise ValueError(
        "No existen registros originales."
    )


# ============================================================
# 4. CALCULAR LOS TRES KPI DE CALIDAD
# ============================================================

# ------------------------------------------------------------
# KPI 1. PORCENTAJE DE REGISTROS VALIDADOS
# ------------------------------------------------------------

# Proporción de registros originales incorporados
# al dataset predictivo.

prv = (
    len(predictivo) / total
) * 100


# ------------------------------------------------------------
# KPI 2. TASA DE INTEGRIDAD DE REGISTROS DE CITAS
# ------------------------------------------------------------

campos = politicas_gobernanza.CAMPOS_OBLIGATORIOS

faltan_columnas = set(campos) - set(datos.columns)

if faltan_columnas:
    raise ValueError(
        f"Faltan columnas: {sorted(faltan_columnas)}"
    )

faltantes = datos[campos].isna().copy()

for campo in campos:

    serie = datos[campo]

    if (
        pd.api.types.is_object_dtype(serie)
        or pd.api.types.is_string_dtype(serie)
    ):

        vacios = (
            serie.astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
        )

        faltantes[campo] = (
            faltantes[campo] | vacios
        )

incompletos = int(
    faltantes.any(axis=1).sum()
)

completos = total - incompletos

integridad = (
    completos / total
) * 100


# ------------------------------------------------------------
# KPI 3. TASA DE DUPLICIDAD DE IDENTIFICADORES
# ------------------------------------------------------------

ids = (
    datos["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

frecuencias = ids.dropna().value_counts()

grupos_repetidos = int(
    (frecuencias > 1).sum()
)

duplicidad = (
    grupos_repetidos / total
) * 100


# ------------------------------------------------------------
# TABLA GENERAL DE KPI
# ------------------------------------------------------------

tabla_kpis_final = pd.DataFrame([
    {
        "Código": "PRV",
        "KPI": "Porcentaje de Registros Validados",
        "Resultado (%)": round(prv, 2),
        "Meta": ">=90%",
        "Estado": (
            "CUMPLE"
            if prv >= 90
            else "NO CUMPLE"
        )
    },
    {
        "Código": "IND-01",
        "KPI": "Tasa de integridad de registros de citas",
        "Resultado (%)": round(integridad, 2),
        "Meta": ">98%",
        "Estado": (
            "CUMPLE"
            if integridad > 98
            else "NO CUMPLE"
        )
    },
    {
        "Código": "IND-05",
        "KPI": "Tasa de duplicidad de identificadores",
        "Resultado (%)": round(duplicidad, 2),
        "Meta": "0%",
        "Estado": (
            "CUMPLE"
            if grupos_repetidos == 0
            else "NO CUMPLE"
        )
    }
])


# ============================================================
# 5. EVALUAR LAS CUATRO POLÍTICAS DE GOBERNANZA
# ============================================================

ruta_catalogos = (
    proyecto / "catalogos_propuestos_gobernanza.json"
)

with open(
    ruta_catalogos,
    encoding="utf-8"
) as archivo:

    catalogos = json.load(archivo)

tabla_politicas_final = (
    politicas_gobernanza.evaluar_politicas(
        datos,
        catalogos=catalogos
    )
)


# ============================================================
# 6. RECUPERAR LAS EVIDENCIAS GUARDADAS
# ============================================================

# ------------------------------------------------------------
# 6.1. PRUEBAS DE LAS CUATRO POLÍTICAS
# ------------------------------------------------------------

tabla_pruebas_final = leer_ultimo_csv(
    "pruebas_politicas_*.csv"
)


# ------------------------------------------------------------
# 6.2. REPORTE DE AUDITORÍA
# ------------------------------------------------------------

auditoria_final = leer_ultimo_csv(
    "auditoria_politicas_*.csv"
)

resumen_auditoria = (
    auditoria_final
    .groupby("politica")
    .agg(
        incidencias=("id_raw", "size"),
        filas_afectadas=("id_raw", "nunique")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 6.3. REGISTRO DE ACCIONES PHVA
# ------------------------------------------------------------

acciones_phva_final = pd.read_csv(
    carpeta_resultados / "registro_acciones_phva.csv"
)


# ------------------------------------------------------------
# 6.4. EVIDENCIA DE LA PRUEBA DEL CICLO PHVA
# ------------------------------------------------------------

prueba_phva_final = leer_ultimo_csv(
    "prueba_phva_pol02_*.csv"
)

# Preparar una copia para su presentación
prueba_phva_final = prueba_phva_final.copy()

# Corregir el nombre de la acción
prueba_phva_final.loc[
    prueba_phva_final["Etapa"] == "HACER",
    "Acción"
] = "Completar campo faltante"

# Mostrar únicamente "Corrección" en el resultado
# de la etapa HACER.
prueba_phva_final.loc[
    prueba_phva_final["Etapa"] == "HACER",
    "Resultado"
] = "Corrección"

# Esta modificación es de presentación.
# El CSV original permanece sin cambios.


# ============================================================
# 7. PRESENTACIÓN FINAL DE LOS RESULTADOS
# ============================================================

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("RESULTADOS FINALES - CALIDAD Y GOBERNANZA DE DATOS")
print("=" * 70)

print("\nREGISTROS EVALUADOS")
print("Datos originales:", total)
print("Dataset predictivo:", len(predictivo))


# ============================================================
# SECCIÓN 1. KPI DE CALIDAD
# ============================================================

print("\n" + "=" * 70)
print("1. TRES KPI DE CALIDAD")
print("=" * 70)

display(tabla_kpis_final)


# ============================================================
# SECCIÓN 2. POLÍTICAS PROGRAMADAS
# ============================================================

print("\n" + "=" * 70)
print("2. POLÍTICAS DE GOBERNANZA PROGRAMADAS")
print("=" * 70)

display(tabla_politicas_final)


# ============================================================
# SECCIÓN 3. PRUEBAS CON VALORES VÁLIDOS E INVÁLIDOS
# ============================================================

print("\n" + "=" * 70)
print("3. PRUEBAS DE LAS CUATRO POLÍTICAS")
print("=" * 70)

display(tabla_pruebas_final)

pruebas_superadas = int(
    (
        tabla_pruebas_final["Resultado prueba"]
        == "SUPERADA"
    ).sum()
)

pruebas_fallidas = (
    len(tabla_pruebas_final) - pruebas_superadas
)

print("\nRESUMEN DE PRUEBAS")

print(
    "Pruebas ejecutadas:",
    len(tabla_pruebas_final)
)

print(
    "Pruebas superadas:",
    pruebas_superadas
)

print(
    "Pruebas fallidas:",
    pruebas_fallidas
)


# ============================================================
# SECCIÓN 4. AUDITORÍA DE INCUMPLIMIENTOS
# ============================================================

print("\n" + "=" * 70)
print("4. AUDITORÍA DE INCUMPLIMIENTOS")
print("=" * 70)

display(resumen_auditoria)

print("\nEJEMPLOS DE HALLAZGOS DE POL-02")

hallazgos_pol02 = auditoria_final[
    auditoria_final["politica"] == "POL-02"
].copy()

display(
    hallazgos_pol02.head(5)
)

print(
    "\nRegistros incompletos identificados:",
    hallazgos_pol02["id_raw"].nunique()
)


# ============================================================
# SECCIÓN 5. REGISTRO DE ACCIONES PHVA
# ============================================================

print("\n" + "=" * 70)
print("5. REGISTRO DE ACCIONES CORRECTIVAS - PHVA")
print("=" * 70)

columnas_phva = [
    "Codigo",
    "Hallazgo",
    "Cantidad",
    "Accion_propuesta",
    "Estado"
]

display(
    acciones_phva_final[columnas_phva]
)


# ============================================================
# SECCIÓN 6. DEMOSTRACIÓN DEL CICLO PHVA
# ============================================================

print("\n" + "=" * 70)
print("6. DEMOSTRACIÓN DEL CICLO PHVA - POL-02")
print("=" * 70)

display(
    prueba_phva_final[
        ["Etapa", "Acción", "Resultado"]
    ]
)


# ============================================================
# 8. RESUMEN GENERAL DE EVIDENCIAS
# ============================================================

print("\n" + "=" * 70)
print("RESUMEN GENERAL DE EVIDENCIAS")
print("=" * 70)

print(
    "KPI calculados:",
    len(tabla_kpis_final)
)

print(
    "Políticas evaluadas:",
    len(tabla_politicas_final)
)

print(
    "Pruebas ejecutadas:",
    len(tabla_pruebas_final)
)

print(
    "Pruebas superadas:",
    pruebas_superadas
)

print(
    "Acciones PHVA registradas:",
    len(acciones_phva_final)
)


# ============================================================
# 9. OBSERVACIONES FINALES
# ============================================================

print("\nOBSERVACIONES:")

print(
    "- Los KPI y las políticas se calcularon "
    "con datos de SQL Server."
)

print(
    "- POL-03 se evaluó utilizando "
    "catálogos provisionales."
)

print(
    "- Las pruebas funcionales se ejecutaron "
    "con registros de prueba."
)

print(
    "- La prueba PHVA corrigió un campo faltante "
    "en una copia de un registro de prueba."
)

print(
    "- Las acciones correctivas sobre los "
    "datos originales siguen pendientes."
)

print(
    "- El PRV expresa la proporción de registros "
    "incorporados al dataset predictivo."
)

print(
    "- No se modificaron los registros "
    "originales de SQL Server."
)

print("\n" + "=" * 70)
print("EVALUACIÓN FINALIZADA")
print("=" * 70)


RED SALUD INTEGRAL CARAZO
RESULTADOS FINALES - CALIDAD Y GOBERNANZA DE DATOS

REGISTROS EVALUADOS
Datos originales: 10080
Dataset predictivo: 8996

1. TRES KPI DE CALIDAD


,Código,KPI,Resultado (%),Meta,Estado
0,PRV,Porcentaje de Registros Validados,89.25,>=90%,NO CUMPLE
1,IND-01,Tasa de integridad de registros de citas,90.87,>98%,NO CUMPLE
2,IND-05,Tasa de duplicidad de identificadores,0.79,0%,NO CUMPLE



2. POLÍTICAS DE GOBERNANZA PROGRAMADAS


,Politica,Control,Incumplimientos,Estado
0,POL-02,Completitud,920,CON HALLAZGOS
1,POL-03,Estandarización,0,SIN HALLAZGOS - CATÁLOGO PROVISIONAL
2,POL-04,Duplicidad,80,REQUIERE REVISIÓN
3,POL-05,Valores anómalos,3,CON HALLAZGOS



3. PRUEBAS DE LAS CUATRO POLÍTICAS


...


Registros incompletos identificados: 920

5. REGISTRO DE ACCIONES CORRECTIVAS - PHVA


,Codigo,Hallazgo,Cantidad,Accion_propuesta,Estado
0,H01,Registros incompletos,920,Revisar los campos faltantes y su captura,PENDIENTE
1,H02,Valores lógicos inválidos,3,Revisar los registros y las reglas incumplidas,PENDIENTE
2,H03,Identificadores repetidos,80,Comprobar los grupos repetidos antes de corregir,PENDIENTE
3,H04,Sin correspondencia predictiva,1084,Investigar por qué no aparecen en el dataset,PENDIENTE



6. DEMOSTRACIÓN DEL CICLO PHVA - POL-02


,Etapa,Acción,Resultado
0,PLANIFICAR,Identificar campo faltante,1 incumplimiento
1,HACER,Completar campo faltante,Corrección
2,VERIFICAR,Repetir POL-02,0 incumplimientos
3,ACTUAR,Comprobar control preventivo,Prueba superada



RESUMEN GENERAL DE EVIDENCIAS
KPI calculados: 3
Políticas evaluadas: 4
Pruebas ejecutadas: 8
Pruebas superadas: 8
Acciones PHVA registradas: 4

OBSERVACIONES:
- Los KPI y las políticas se calcularon con datos de SQL Server.
- POL-03 se evaluó utilizando catálogos provisionales.
- Las pruebas funcionales se ejecutaron con registros de prueba.
- La prueba PHVA corrigió un campo faltante en una copia de un registro de prueba.
- Las acciones correctivas sobre los datos originales siguen pendientes.
- El PRV expresa la proporción de registros incorporados al dataset predictivo.
- No se modificaron los registros originales de SQL Server.

EVALUACIÓN FINALIZADA


In [5]:

# ============================================================
# PASO 40. EDA Y REGISTRO DE HALLAZGOS
# RED SALUD INTEGRAL CARAZO
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from sqlalchemy import text
from IPython.display import display

# Ubicar el proyecto
rutas = [
    Path.cwd(),
    Path.cwd() / "RedSaludCarazo_proyecto_base",
    Path.cwd().parent
]

proyecto = next(
    (p for p in rutas if (p / "resultados_jupyter").is_dir()),
    None
)

if proyecto is None:
    raise FileNotFoundError(
        "No se encontró resultados_jupyter."
    )

carpeta = proyecto / "resultados_jupyter"

fecha = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

# ============================================================
# 1. LEER LOS DATOS ORIGINALES
# ============================================================

with engine.connect() as conexion:
    datos_eda = pd.read_sql_query(
        text("SELECT * FROM dbo.raw_cita"),
        conexion
    )

print("=" * 65)
print("1. ANÁLISIS EXPLORATORIO DE DATOS - EDA")
print("=" * 65)

print("Registros:", len(datos_eda))
print("Columnas:", len(datos_eda.columns))

# ============================================================
# 2. ANALIZAR COMPLETITUD POR CAMPO
# ============================================================

perfil = []

for campo in datos_eda.columns:

    serie = datos_eda[campo]

    faltante = serie.isna()

    if (
        pd.api.types.is_object_dtype(serie)
        or pd.api.types.is_string_dtype(serie)
    ):
        vacio = (
            serie.astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
        )
        faltante = faltante | vacio

    cantidad = int(faltante.sum())

    perfil.append({
        "Campo": campo,
        "Tipo": str(serie.dtype),
        "Valores_faltantes": cantidad,
        "Porcentaje_faltante": round(
            cantidad / len(datos_eda) * 100, 2
        ),
        "Valores_distintos": int(serie.nunique())
    })

tabla_eda = pd.DataFrame(perfil)

print("\nCOMPLETITUD POR CAMPO")

display(
    tabla_eda.sort_values(
        "Valores_faltantes",
        ascending=False
    )
)

# ============================================================
# 3. EXPLORAR VARIABLES NUMÉRICAS
# ============================================================

columnas_numericas = [
    "espera_dias",
    "citas_previas_perdidas",
    "recordatorios_enviados"
]

numericas = datos_eda[columnas_numericas].apply(
    pd.to_numeric,
    errors="coerce"
)

print("\nESTADÍSTICAS DE VARIABLES NUMÉRICAS")
display(numericas.describe().T)

# ============================================================
# 4. EXPLORAR DUPLICIDAD
# ============================================================

ids = (
    datos_eda["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

frecuencias = ids.dropna().value_counts()

grupos_repetidos = int(
    (frecuencias > 1).sum()
)

filas_repetidas = int(
    frecuencias[frecuencias > 1].sum()
)

print("\nDUPLICIDAD")
print("Identificadores repetidos:", grupos_repetidos)
print("Filas involucradas:", filas_repetidas)

# ============================================================
# 5. DISTRIBUCIÓN DE ASISTENCIA
# ============================================================

print("\nDISTRIBUCIÓN DE NO_ASISTIO")

distribucion = (
    datos_eda["no_asistio"]
    .astype("string")
    .str.strip()
    .fillna("SIN DATO")
    .value_counts()
    .reset_index()
)

distribucion.columns = [
    "Valor",
    "Cantidad"
]

display(distribucion)

# ============================================================
# 6. REGISTRO DE HALLAZGOS
# ============================================================

archivos = list(
    carpeta.glob("auditoria_politicas_*.csv")
)

if not archivos:
    raise FileNotFoundError(
        "No se encontró el reporte de auditoría."
    )

archivo_auditoria = max(
    archivos,
    key=lambda p: p.stat().st_mtime
)

registro_hallazgos = pd.read_csv(
    archivo_auditoria
)

registro_hallazgos["Fuente"] = "dbo.raw_cita"

registro_hallazgos["Estado"] = "DETECTADO"

registro_hallazgos["Fecha_revision_UTC"] = (
    datetime.now(timezone.utc).isoformat()
)

print("\n" + "=" * 65)
print("2. REGISTRO DE HALLAZGOS")
print("=" * 65)

resumen_hallazgos = (
    registro_hallazgos
    .groupby(["politica", "campo"])
    .agg(
        Incidencias=("id_raw", "size"),
        Registros_afectados=("id_raw", "nunique")
    )
    .reset_index()
)

display(resumen_hallazgos)

print("\nEJEMPLOS DE HALLAZGOS")
display(registro_hallazgos.head(10))

# ============================================================
# 7. GUARDAR EVIDENCIAS
# ============================================================

tabla_eda.to_csv(
    carpeta / f"eda_perfil_{fecha}.csv",
    index=False,
    encoding="utf-8-sig"
)

registro_hallazgos.to_csv(
    carpeta / f"registro_hallazgos_{fecha}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivos generados:")
print(f"- eda_perfil_{fecha}.csv")
print(f"- registro_hallazgos_{fecha}.csv")
print("\nNo se modificaron los registros originales.")


1. ANÁLISIS EXPLORATORIO DE DATOS - EDA
Registros: 10080
Columnas: 19

COMPLETITUD POR CAMPO


,Campo,Tipo,Valores_faltantes,Porcentaje_faltante,Valores_distintos
8,especialidad,str,120,1.19,10
5,centro_salud,str,120,1.19,12
6,canal_agendamiento,str,120,1.19,8
10,espera_dias,float64,120,1.19,62
11,recordatorios_enviados,float64,120,1.19,5
12,citas_previas_perdidas,float64,120,1.19,6
7,grupo_edad,str,120,1.19,4
15,no_asistio,str,80,0.79,2
2,origen_dataset,str,0,0.00,1
0,id_raw,int64,0,0.00,10080



ESTADÍSTICAS DE VARIABLES NUMÉRICAS


,count,mean,std,min,25%,50%,75%,max
espera_dias,9960.0,19.883032,16.152777,-1.0,7.0,15.0,29.0,60.0
citas_previas_perdidas,9960.0,1.411245,1.265353,-1.0,0.0,1.0,2.0,4.0
recordatorios_enviados,9960.0,1.132631,1.085639,-1.0,0.0,1.0,2.0,3.0



DUPLICIDAD
Identificadores repetidos: 80
Filas involucradas: 160

DISTRIBUCIÓN DE NO_ASISTIO


...


2. REGISTRO DE HALLAZGOS


,politica,campo,Incidencias,Registros_afectados
0,POL-02,canal_agendamiento,120,120
1,POL-02,centro_salud,120,120
2,POL-02,citas_previas_perdidas,120,120
3,POL-02,especialidad,120,120
4,POL-02,espera_dias,120,120
5,POL-02,grupo_edad,120,120
6,POL-02,no_asistio,80,80
7,POL-02,recordatorios_enviados,120,120
8,POL-04,id_cita,160,160
9,POL-05,citas_previas_perdidas,1,1



EJEMPLOS DE HALLAZGOS


,id_raw,politica,campo,hallazgo,Fuente,Estado,Fecha_revision_UTC
0,38,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
1,330,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
2,511,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
3,556,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
4,673,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
5,675,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
6,680,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
7,683,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
8,766,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00
9,795,POL-02,centro_salud,Campo obligatorio faltante,dbo.raw_cita,DETECTADO,2026-10-22T00:44:52.093481+00:00



Archivos generados:
- eda_perfil_20261022T004451Z.csv
- registro_hallazgos_20261022T004451Z.csv

No se modificaron los registros originales.
